# CircuitVision · Train a laboratory-component detector

**Student-friendly notebook: no coding required.** It downloads and combines two publicly licensed, annotated object-detection datasets, then trains a YOLO11 model for electronics parts like **Resistor, LED, LDR, Power Module, Capacitor, Breadboard**, and more.

**Before pressing Run all:** Go to **Runtime → Change runtime type → T4 GPU** (if available). You will also need a *personal* [Roboflow API key](https://app.roboflow.com/settings/api) so the notebook can download the public datasets. **Never post your key in a chat, screenshot, GitHub, or shared notebook.** You'll be asked to enter it privately.

Data sources (not authored by the CircuitVision team):
- [EESOB, Roboflow Universe, version 4](https://universe.roboflow.com/android-yolo/eesob/dataset/4), CC BY 4.0 as listed by publisher
- [Electronics Components SVECk, version 7](https://universe.roboflow.com/vishnupriyas-workspace/electronics-components-sveck/dataset/7), CC BY 4.0 as listed by publisher

Keep these credits in your research paper. This notebook will **train a new model** from these third-party datasets; it is distinct from the current pretrained EESOB model already on the website. Dataset content/availability may be changed by the publishers. Expect dataset downloads to use substantial time and storage, and training time to vary with Colab GPU availability.

**What happens:** (1) Install tools → (2) Download both sets → (3) Merge labels safely → (4) Check classes → (5) Train → (6) Evaluate and download `best.pt`, `best.onnx`, and `classes.json`.

## 1 · Install the tools

Press **Run all** in the Runtime menu. The cells execute in order. The notebook will stop with an understandable error if a dataset can't be downloaded or has incorrect labels.

In [ ]:
%pip -q install "ultralytics>=8.3,<9" "roboflow>=1.1" "PyYAML>=6" "onnx>=1.15" "onnxruntime>=1.18" "onnxslim>=0.1"
import torch
print('GPU:', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'NOT FOUND — select Runtime → Change runtime type → T4 GPU')
assert torch.cuda.is_available(), 'Please enable a GPU in Colab before training.'

## 2 · Download the two annotated datasets

When prompted, paste **your Roboflow API key** into the hidden input field and press Enter. It is used only to access Roboflow during this session and is not saved in the notebook. If the dataset provider restricts access, visit its Roboflow page and follow the download/fork instructions shown there.

In [ ]:
from getpass import getpass
from pathlib import Path
from roboflow import Roboflow
import os, gc

api_key = getpass('Roboflow API key (hidden): ').strip()
assert api_key, 'Please provide your own Roboflow API key.'
rf = Roboflow(api_key=api_key)
DATA_ROOT = Path('/content/circuitvision_sources')
DATA_ROOT.mkdir(parents=True, exist_ok=True)

def get_source(workspace, project, version, local_folder):
    dest = DATA_ROOT / local_folder
    existing = list(dest.rglob('data.yaml')) if dest.exists() else []
    if existing:
        print(f'Reusing {local_folder} dataset:', existing[0].parent)
        return existing[0].parent
    try:
        exported = rf.workspace(workspace).project(project).version(version).download('yolov11',location=str(dest))
    except Exception as exc:
        raise RuntimeError(f'Could not download {workspace}/{project}/{version}. Check your Roboflow account and project access. Original error: {exc}') from exc
    return Path(exported.location)

sources = {
    'eesob': get_source('android-yolo', 'eesob', 4, 'eesob_v4'),
    'components': get_source('vishnupriyas-workspace', 'electronics-components-sveck', 7, 'components_v7'),
}
del rf, api_key
print('Both datasets downloaded:', sources)

## 3 · Combine annotations properly

A number such as `0` may refer to different objects in different datasets. This cell maps each source label by its **class name** and writes new, consistent YOLO bounding-box annotations. It preserves existing train/validation/test splits, skips byte-identical duplicates across splits, and does not invent boxes.

**Quick training** is recommended for your first run (roughly a few thousand images). It checks that everything works without spending hours on a full run. For a larger final experiment, change `QUICK_RUN` to `False`, then rerun the notebook on a GPU. A full training run can take many hours and Colab may disconnect.

**Limitation:** Images from one dataset may contain other parts that were not labeled in that dataset. Combining two sources does not automatically annotate those missing objects. The final model still needs manual evaluation on your school's actual components.

In [ ]:
# Training mode — you only need to change this one value for a bigger run later.
QUICK_RUN = True
MAX_PER_SPLIT = {'train': 2200, 'val': 250, 'test': 250} if QUICK_RUN else {}

# The helper is embedded here so you do not need a separate Python file.

In [ ]:
"""Merge Roboflow YOLO object-detection exports without class-ID collisions.

Designed for CircuitVision's EESOB + Electronics Components datasets.
No labels are fabricated; missing label files count as empty images.
"""
from __future__ import annotations

import hashlib
import random
import re
import shutil
from collections import Counter
from dataclasses import dataclass
from pathlib import Path

import yaml

IMAGE_EXTS = {'.jpg', '.jpeg', '.png', '.webp', '.bmp'}
# Preserve notable electronics nomenclature in result labels.
DISPLAY = {
    'led': 'LED', 'ldr': 'LDR', 'led matrix': 'LED Matrix',
    'ic': 'IC', 'lcd': 'LCD', 'lcd oled': 'LCD OLED', 'rfid': 'RFID',
    'rtc': 'RTC', 'ir sensor': 'IR Sensor', 'ir receiver': 'IR Receiver',
    'esp32': 'ESP32', 'esp wifi': 'ESP WiFi',
    'arduino uno': 'Arduino Uno', 'arduino nano': 'Arduino Nano',
    'arduino mini': 'Arduino Mini', 'arduino mega': 'Arduino Mega',
    'battery 9v': 'Battery 9V', '9v battery': 'Battery 9V',
    'power module': 'Power Module', 'resistor': 'Resistor',
    'capacitor': 'Capacitor', 'potentiometer': 'Potentiometer',
    'prototype pcb': 'Prototype PCB', '7 segment': '7-segment',
}


def class_key(value: str) -> str:
    normalized = re.sub(r'[_\-]+', ' ', str(value).strip().lower())
    normalized = re.sub(r'\s+', ' ', normalized).strip()
    if not normalized:
        raise ValueError('Found blank class name')
    return normalized


def display_name(key: str) -> str:
    return DISPLAY.get(key, key.title())


def read_class_names(folder: Path) -> list[str]:
    yaml_path = folder / 'data.yaml'
    if not yaml_path.is_file():
        matches = list(folder.rglob('data.yaml'))
        if len(matches) != 1:
            raise FileNotFoundError(f'Expected one data.yaml under {folder}; found {len(matches)}')
        yaml_path = matches[0]
    obj = yaml.safe_load(yaml_path.read_text(encoding='utf-8'))
    names = obj.get('names')
    if isinstance(names, dict):
        ids = sorted(int(k) for k in names)
        if ids != list(range(len(ids))):
            raise ValueError(f'Class IDs in {yaml_path} are not contiguous starting at zero')
        names = [str(names[i] if i in names else names[str(i)]) for i in ids]
    if not isinstance(names, list) or not names:
        raise ValueError(f'Invalid or missing class list in {yaml_path}')
    keys = [class_key(n) for n in names]
    if len(set(keys)) != len(keys):
        raise ValueError(f'Repeated normalized class names in {yaml_path}: {names}')
    return [str(n) for n in names]


def split_images(folder: Path, split: str) -> Path | None:
    labels = ('valid', 'val') if split == 'val' else (split,)
    for part in labels:
        dest = folder / part / 'images'
        if dest.is_dir():
            return dest
    # Some Roboflow packages have an extra top-level directory.
    matches = [p for p in folder.rglob('images') if p.parent.name in labels and p.is_dir()]
    if len(matches) == 1:
        return matches[0]
    if matches:
        raise ValueError(f'Ambiguous {split} image directories under {folder}')
    return None


@dataclass
class Sample:
    path: Path
    labels: Path
    source: str
    split: str
    local_names: list[str]


def read_rows(path: Path, class_count: int) -> list[tuple[int, list[float]]]:
    if not path.exists():
        return []
    parsed = []
    for line_number, line in enumerate(path.read_text(encoding='utf-8').splitlines(), start=1):
        if not line.strip():
            continue
        parts = line.split()
        if len(parts) != 5:
            raise ValueError(f'{path}:{line_number}: expected YOLO class x y w h, got {len(parts)} values')
        try:
            raw_id = float(parts[0])
            coordinates = [float(x) for x in parts[1:]]
        except ValueError as exc:
            raise ValueError(f'{path}:{line_number}: not numeric') from exc
        local_id = int(raw_id)
        if raw_id != local_id or not 0 <= local_id < class_count:
            raise ValueError(f'{path}:{line_number}: class {parts[0]} not in 0..{class_count-1}')
        if any(not 0 <= value <= 1 for value in coordinates) or coordinates[2] <= 0 or coordinates[3] <= 0:
            raise ValueError(f'{path}:{line_number}: invalid normalized box {coordinates}')
        parsed.append((local_id, coordinates))
    return parsed


def source_samples(folder: Path, source_name: str):
    names = read_class_names(folder)
    result = {'train': [], 'val': [], 'test': []}
    for split in result:
        image_dir = split_images(folder, split)
        if image_dir is None:
            continue
        label_dir = image_dir.parent / 'labels'
        for img in sorted(image_dir.iterdir()):
            if img.is_file() and img.suffix.lower() in IMAGE_EXTS:
                result[split].append(Sample(img, label_dir / f'{img.stem}.txt', source_name, split, names))
    if not result['train'] or not result['val']:
        raise ValueError(f'{source_name}: both train and validation images are required')
    return result, names


def train_merge(roots: dict[str, Path], out: Path, limits: dict[str, int | None] | None = None,
                seed: int = 42) -> dict:
    """Create combined YOLO dataset. Respect source train/val/test splits.

    limits controls an optional quick-training cap per source/split, but does not
    change which split an image belongs to. All box class IDs are remapped.
    """
    if out.exists():
        raise FileExistsError(f'Remove old combined directory before rerunning: {out}')
    rng = random.Random(seed)
    sources = []
    for name, root in roots.items():
        samples, names = source_samples(Path(root), name)
        sources.append((name, samples, names))
    keys = sorted({class_key(n) for _name, _samples, names in sources for n in names})
    key_id = {key: ix for ix, key in enumerate(keys)}
    names = [display_name(key) for key in keys]
    if len(set(names)) != len(names):
        raise ValueError('Normalized display class names have duplicate values')
    counts = Counter()
    images = Counter()
    skipped_duplicates = 0
    missing_labels = 0
    seen_hashes = set()
    # Reserve holdout images first, to avoid leaking identical files into training.
    for split in ('test', 'val', 'train'):
        for source_name, samples, local_names in sources:
            candidates = list(samples[split])
            rng.shuffle(candidates)
            limit = (limits or {}).get(split)
            if limit is not None:
                limit = int(limit)
                if limit < 1:
                    raise ValueError(f'Sample limit for {split} must be positive')
                if split == 'train':
                    # Small training samples should not accidentally omit rare,
                    # important lab components. Favor a handful of annotated
                    # examples for each priority class before random sampling.
                    priority = {'resistor', 'led', 'ldr', 'power module',
                                'capacitor', 'breadboard', 'potentiometer'}
                    by_class = {name: [] for name in priority}
                    for image in candidates:
                        present = {class_key(local_names[i]) for i, _ in
                                   read_rows(image.labels, len(local_names))}
                        for name in present.intersection(priority):
                            by_class[name].append(image)
                    priority_images = []
                    selected = set()
                    for name in sorted(priority):
                        for image in by_class[name][:12]:
                            if image.path not in selected:
                                priority_images.append(image)
                                selected.add(image.path)
                    candidates = priority_images[:limit] + [
                        image for image in candidates if image.path not in selected
                    ][:max(0, limit - len(priority_images))]
                else:
                    candidates = candidates[:limit]
            for image in candidates:
                digest = hashlib.sha256(image.path.read_bytes()).hexdigest()
                if digest in seen_hashes:
                    skipped_duplicates += 1
                    continue
                seen_hashes.add(digest)
                rows = read_rows(image.labels, len(local_names))
                if not image.labels.is_file():
                    missing_labels += 1
                # Stable source prefix and digest prevent filename collisions.
                filename = f'{source_name}_{digest[:16]}{image.path.suffix.lower()}'
                dest_img = out / split / 'images' / filename
                dest_lbl = out / split / 'labels' / (Path(filename).stem + '.txt')
                dest_img.parent.mkdir(parents=True, exist_ok=True)
                dest_lbl.parent.mkdir(parents=True, exist_ok=True)
                shutil.copy2(image.path, dest_img)
                with dest_lbl.open('w', encoding='utf-8') as f:
                    for local_id, box in rows:
                        new_id = key_id[class_key(local_names[local_id])]
                        f.write(str(new_id) + ' ' + ' '.join(f'{x:.6f}' for x in box) + '\n')
                        counts[(split, names[new_id])] += 1
                images[(split, source_name)] += 1
    if not sum(c for (s,_),c in images.items() if s == 'train') or not sum(c for (s,_),c in images.items() if s=='val'):
        raise ValueError('Combined dataset missing train or validation split')
    yaml_data = {'path': str(out.resolve()), 'train': 'train/images', 'val': 'val/images',
                 'test': 'test/images' if sum(c for (s,_),c in images.items() if s=='test') else 'val/images',
                 'nc': len(names), 'names': names}
    (out / 'data.yaml').write_text(yaml.safe_dump(yaml_data, sort_keys=False, allow_unicode=True), encoding='utf-8')
    return {'names': names, 'image_counts': dict(images), 'box_counts': dict(counts),
            'duplicate_images_skipped': skipped_duplicates, 'images_without_label_files': missing_labels,
            'yaml_path': str(out / 'data.yaml')}

In [ ]:
import shutil
from pathlib import Path
COMBINED_ROOT = Path('/content/circuitvision_combined')
if COMBINED_ROOT.exists():
    shutil.rmtree(COMBINED_ROOT)
report = train_merge(sources, COMBINED_ROOT, limits=MAX_PER_SPLIT, seed=42)
print('Merged classes:', len(report['names']))
print('Image counts by original source:', report['image_counts'])
print('Byte-identical duplicate images excluded:', report['duplicate_images_skipped'])
print('Images with missing label files (treated as backgrounds):', report['images_without_label_files'])
print('Merged configuration:', report['yaml_path'])

## 4 · Check the important electronics classes

Make sure the combined dataset actually contains training boxes for the parts we care about. A class being **listed** does not guarantee high-quality detection; this is a basic annotation audit.

In [ ]:
from collections import Counter
important = ['Resistor','LED','LDR','Power Module','Capacitor','Breadboard','Potentiometer','Arduino Uno']
training_boxes = Counter({name: count for (split, name), count in report['box_counts'].items() if split == 'train'})
for label in important:
    print(f'{label:<18} {training_boxes.get(label, 0):>6} training boxes')
missing = [label for label in ['Resistor','LED','LDR','Power Module'] if training_boxes.get(label, 0)==0]
assert not missing, f'No training boxes found for {missing}. Check source dataset versions or increase sample size.'
print('✓ Priority classes have bounding-box annotations.')

## 5 · Train your own YOLO11 detector

The notebook starts from published YOLO11 pretrained weights and **fine-tunes a new CircuitVision model** on the combined laboratory dataset. Model training may take a while. A T4 GPU and adequate free runtime must be available.

- Quick run: small YOLO11n model, 25 epochs, 512-pixel images.
- Larger run: YOLO11s, up to 60 epochs, 640-pixel images.

Neither run guarantees any specific accuracy. The next cell measures results on reserved test images.

In [ ]:
from ultralytics import YOLO
from pathlib import Path
model_name = 'yolo11n.pt' if QUICK_RUN else 'yolo11s.pt'
epochs = 25 if QUICK_RUN else 60
image_size = 512 if QUICK_RUN else 640
model = YOLO(model_name)
model.train(
    data=report['yaml_path'], epochs=epochs, imgsz=image_size,
    batch=8, device=0, workers=2, patience=12, seed=42,
    project='/content/circuitvision_runs', name='combined', exist_ok=True,
    amp=True, save=True, plots=True,
)
BEST = Path('/content/circuitvision_runs/combined/weights/best.pt')
assert BEST.is_file(), 'Training did not finish or save best.pt; check the output above.'
print('Model trained and saved:', BEST)

## 6 · Test and export for the CircuitVision website

This runs a real evaluation on the reserved test images (when provided; otherwise on validation data). It exports your newly trained model for the Render website's CPU-friendly ONNX runtime, alongside the exact class-name mapping. **The notebook does not upload model files into GitHub or Render automatically.**

In [ ]:
from ultralytics import YOLO
from pathlib import Path
import json, shutil, zipfile
trained = YOLO(str(BEST))
metrics = trained.val(data=report['yaml_path'], split='test', imgsz=image_size, device=0, plots=False)
print('Measured test mAP50:', round(float(metrics.box.map50), 4))
print('Measured test mAP50-95:', round(float(metrics.box.map), 4))

onnx_path = Path(trained.export(format='onnx', imgsz=640, opset=17,
                                dynamic=False, nms=False, simplify=True))
result_dir=Path('/content/CircuitVision_Trained_Model')
result_dir.mkdir(exist_ok=True)
shutil.copy2(BEST,result_dir/'best.pt')
shutil.copy2(onnx_path,result_dir/'best.onnx')
class_map={str(i): name for i,name in enumerate(report['names'])}
(result_dir/'classes.json').write_text(json.dumps(class_map, indent=2),encoding='utf-8')
(result_dir/'DATASET_SOURCES.txt').write_text(
    'EESOB: https://universe.roboflow.com/android-yolo/eesob/dataset/4 (CC BY 4.0)\n'
    'Electronics Components: https://universe.roboflow.com/vishnupriyas-workspace/electronics-components-sveck/dataset/7 (CC BY 4.0)\n'
    'Model: fine-tuned by CircuitVision team using the datasets above.\n'
    'Do not report any evaluation values that were not actually measured.\n'
)
zip_path=Path('/content/CircuitVision_Trained_Model.zip')
with zipfile.ZipFile(zip_path,'w',zipfile.ZIP_DEFLATED) as z:
    for file in result_dir.iterdir(): z.write(file,arcname=file.name)
print('SUCCESS: trained model package ready:',zip_path)
print('Contents:', [f.name for f in result_dir.iterdir()])

## 7 · Download the trained model (final step)

When Colab finishes all the steps, your browser downloads the ZIP. **Upload that ZIP back to your CircuitVision ChatGPT conversation** so the website can be updated to use your team's own model. Do not upload Roboflow API keys.

If automatic downloads don't work on mobile, look for the ZIP under the **Files** folder in Colab's left sidebar or run the next cell again on a laptop.

In [ ]:
from google.colab import files
files.download('/content/CircuitVision_Trained_Model.zip')